# Fraud Detection – Model Training

Trains 5 classifiers across 3 feature sets with optimised hyperparameters matching the README specification.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import os, joblib, time, warnings
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.base import clone
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix, roc_curve
)

warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
print('Libraries loaded.')


## 2. Load Feature Sets

In [ ]:
y_train = pd.read_parquet('../data/processed/y_train.parquet').iloc[:, 0]
y_test  = pd.read_parquet('../data/processed/y_test.parquet').iloc[:, 0]

X_train_full     = pd.read_parquet('../data/processed/X_train_full.parquet')
X_test_full      = pd.read_parquet('../data/processed/X_test_full.parquet')
X_train_selected = pd.read_parquet('../data/processed/X_train_selected.parquet')
X_test_selected  = pd.read_parquet('../data/processed/X_test_selected.parquet')
X_train_pca      = pd.read_parquet('../data/processed/X_train_pca.parquet')
X_test_pca       = pd.read_parquet('../data/processed/X_test_pca.parquet')

print(f'Full feature set    : {X_train_full.shape}')
print(f'Selected feature set: {X_train_selected.shape}')
print(f'PCA feature set     : {X_train_pca.shape}')


## 3. Compute Class Imbalance Weight

**Fix:** compute `scale_pos_weight` from actual training labels instead of hardcoding.

In [ ]:
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_pos_weight = neg_count / pos_count

print(f'Negative (legit) samples : {neg_count:,}')
print(f'Positive (fraud) samples : {pos_count:,}')
print(f'scale_pos_weight          : {scale_pos_weight:.4f}')


## 4. Define Optimised Models

**Fixes vs original:**
- **XGBoost**: `learning_rate=0.05`, `n_estimators=300`, `max_depth=8` (was 0.1/200/6)
- **LightGBM**: `learning_rate=0.05`, `n_estimators=200`, `num_leaves=47`, `max_depth=8` (was 0.1/150/31/6)
- **RandomForest**: `n_estimators=200`, `max_depth=15`, `min_samples_leaf=5` (was 100/12/4)

In [ ]:
models = {
    'LogisticRegression': LogisticRegression(
        class_weight='balanced',
        max_iter=2000,
        random_state=SEED,
        solver='saga',
        C=0.1,
        tol=1e-4
    ),
    'DecisionTree': DecisionTreeClassifier(
        class_weight='balanced',
        random_state=SEED,
        max_depth=12,
        min_samples_split=10,
        min_samples_leaf=4
    ),
    # FIX: 200 estimators, max_depth=15, min_leaf=5 (README spec)
    'RandomForest': RandomForestClassifier(
        class_weight='balanced',
        random_state=SEED,
        n_jobs=-1,
        n_estimators=200,
        max_depth=15,
        max_features='sqrt',
        min_samples_leaf=5
    ),
    # FIX: LR=0.05, 300 estimators, max_depth=8 (README spec)
    'XGBoost': XGBClassifier(
        scale_pos_weight=scale_pos_weight,
        eval_metric='logloss',
        random_state=SEED,
        n_jobs=-1,
        tree_method='hist',
        n_estimators=300,
        max_depth=8,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.5,
        min_child_weight=3,
        gamma=0.1
    ),
    # FIX: LR=0.05, 200 estimators, num_leaves=47, max_depth=8 (README spec)
    'LightGBM': LGBMClassifier(
        scale_pos_weight=scale_pos_weight,
        random_state=SEED,
        verbose=-1,
        n_jobs=-1,
        n_estimators=200,
        num_leaves=47,
        max_depth=8,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_samples=20,
        reg_alpha=0.1,
        reg_lambda=1.5
    )
}
print('Models defined:')
for name in models:
    print(f'  {name}')


## 5. Training Pipeline

In [ ]:
feature_sets = {
    'Full'    : (X_train_full,     X_test_full),
    'Selected': (X_train_selected, X_test_selected),
    'PCA'     : (X_train_pca,      X_test_pca),
}

results_list  = []
fitted_models = {}

for model_name, model_template in models.items():
    print(f'\n{"="*55}')
    print(f'  Model: {model_name}')
    print(f'{"="*55}')
    for fs_name, (X_tr, X_te) in feature_sets.items():
        print(f'  [{fs_name:8s}] training on {X_tr.shape[1]} features...', end=' ', flush=True)
        model = clone(model_template)
        t0 = time.time()
        try:
            model.fit(X_tr, y_train)
        except Exception as e:
            print(f'FAILED: {e}')
            continue
        fit_time = time.time() - t0

        model_id = f'{model_name}_{fs_name}'
        fitted_models[model_id] = model

        y_pred = model.predict(X_te)
        y_prob = model.predict_proba(X_te)[:, 1]

        auc  = roc_auc_score(y_test, y_prob)
        acc  = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, zero_division=0)
        rec  = recall_score(y_test, y_pred, zero_division=0)
        f1   = f1_score(y_test, y_pred, zero_division=0)

        results_list.append({
            'Model': model_name, 'Feature Set': fs_name,
            'AUC': round(auc, 4), 'Accuracy': round(acc, 4),
            'Precision': round(prec, 4), 'Recall': round(rec, 4),
            'F1': round(f1, 4), 'TrainTime_sec': round(fit_time, 1)
        })
        print(f'done ({fit_time:.1f}s)  AUC={auc:.4f}  F1={f1:.4f}')

df_results = pd.DataFrame(results_list)
print('\nAll models trained.\n')
display(df_results.sort_values('AUC', ascending=False).reset_index(drop=True))


## 6. Performance Comparison Plots

In [ ]:
os.makedirs('../results/figures', exist_ok=True)

# AUC heatmap across models × feature sets
pivot_auc = df_results.pivot(index='Model', columns='Feature Set', values='AUC')
plt.figure(figsize=(9, 5))
sns.heatmap(pivot_auc, annot=True, fmt='.4f', cmap='YlGn', linewidths=0.3)
plt.title('AUC-ROC Heatmap: Model × Feature Set')
plt.tight_layout()
plt.savefig('../results/figures/auc_heatmap.png', dpi=150)
plt.show()

# Grouped bar chart: AUC by model & feature set
plt.figure(figsize=(13, 6))
sns.barplot(data=df_results, x='Model', y='AUC', hue='Feature Set', palette='viridis')
plt.title('AUC-ROC Across Models and Feature Sets')
plt.ylim(0.5, 1.0)
plt.legend(bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.savefig('../results/figures/model_comparison.png', dpi=150)
plt.show()

# Multi-metric comparison
df_melted = df_results.melt(
    id_vars=['Model', 'Feature Set'],
    value_vars=['AUC', 'F1', 'Recall', 'Precision'],
    var_name='Metric', value_name='Score'
)
g = sns.FacetGrid(df_melted, col='Metric', col_wrap=2, height=4, sharey=False)
g.map_dataframe(sns.barplot, x='Model', y='Score', hue='Feature Set', palette='Set2')
g.add_legend()
for ax in g.axes.flat:
    ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig('../results/figures/all_metrics_comparison.png', dpi=150)
plt.show()


## 7. ROC Curves – Top Models

In [ ]:
best_models = ['XGBoost', 'LightGBM', 'RandomForest']
colors     = {'Full': 'blue', 'Selected': 'green', 'PCA': 'red'}
linestyles = {'XGBoost': '-', 'LightGBM': '--', 'RandomForest': ':'}

plt.figure(figsize=(10, 8))
for model_name in best_models:
    for fs_name, (_, X_te) in feature_sets.items():
        mid = f'{model_name}_{fs_name}'
        if mid not in fitted_models:
            continue
        y_prob = fitted_models[mid].predict_proba(X_te)[:, 1]
        fpr, tpr, _ = roc_curve(y_test, y_prob)
        auc_val = roc_auc_score(y_test, y_prob)
        plt.plot(fpr, tpr,
                 color=colors[fs_name],
                 linestyle=linestyles[model_name],
                 label=f'{model_name} ({fs_name}) AUC={auc_val:.3f}')

plt.plot([0, 1], [0, 1], 'k--', alpha=0.4)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves: Top 3 Models × Feature Sets')
plt.legend(loc='lower right', fontsize=8)
plt.tight_layout()
plt.savefig('../results/figures/roc_curves.png', dpi=150)
plt.show()


## 8. Confusion Matrices

In [ ]:
for model_name in ['RandomForest', 'XGBoost']:
    for fs_name in ['Full', 'Selected', 'PCA']:
        mid = f'{model_name}_{fs_name}'
        if mid not in fitted_models:
            continue
        y_pred = fitted_models[mid].predict(feature_sets[fs_name][1])
        cm = confusion_matrix(y_test, y_pred)

        fig, ax = plt.subplots(figsize=(5, 4))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax)
        ax.set_title(f'{model_name} ({fs_name})')
        ax.set_xlabel('Predicted')
        ax.set_ylabel('Actual')
        plt.tight_layout()
        plt.savefig(f'../results/figures/cm_{model_name.lower()}_{fs_name.lower()}.png', dpi=150)
        plt.close()
print('Confusion matrices saved.')


## 9. Save Metrics & Models

In [ ]:
os.makedirs('../results/metrics', exist_ok=True)
df_results.to_csv('../results/metrics/metrics_summary.csv', index=False)
print('Metrics saved to results/metrics/metrics_summary.csv')

os.makedirs('../models', exist_ok=True)
for mid, model in fitted_models.items():
    joblib.dump(model, f'../models/{mid.lower()}.pkl')
print(f'Saved {len(fitted_models)} model files to models/')

# Print best model summary
best_row = df_results.loc[df_results['AUC'].idxmax()]
print(f'\nBEST MODEL: {best_row["Model"]} on {best_row["Feature Set"]} features')
print(f'  AUC={best_row["AUC"]:.4f}  F1={best_row["F1"]:.4f}  Recall={best_row["Recall"]:.4f}')


## Summary of Fixes Applied

| Parameter | Old value | Fixed value | Impact |
|---|---|---|---|
| scale_pos_weight | hardcoded 27.58 | computed from y_train | accurate for any split |
| XGBoost LR | 0.1 | 0.05 | better generalisation |
| XGBoost n_estimators | 200 | 300 | stronger ensemble |
| XGBoost max_depth | 6 | 8 | more complex patterns |
| LightGBM LR | 0.1 | 0.05 | better generalisation |
| LightGBM num_leaves | 31 | 47 | more expressive trees |
| LightGBM max_depth | 6 | 8 | deeper trees |
| RandomForest n_estimators | 100 | 200 | more stable ensemble |
| RandomForest max_depth | 12 | 15 | captures more patterns |
| RandomForest min_leaf | 4 | 5 | slight regularisation |
